# 🛒 Retail Product Recommendations
### แนะนำสินค้าที่ซื้อคู่กัน เพื่อจัดโปรโมชันและจัดวางสินค้าในร้าน
CP020003 Artificial Intelligence — Final Project

ใช้โมเดลตามที่เรียนใน Week 7: **Apriori**, **FP-Growth** และ **Item2Vec**
กด `Runtime → Run all` ได้เลย

## 1. ติดตั้งและ import

In [ ]:
!pip -q install mlxtend gensim

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # ซ่อนข้อความเตือน DeprecationWarning ของ Colab (ไม่กระทบผลลัพธ์)

import os, time, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
from gensim.models import Word2Vec

## 2. ดึงข้อมูลลง Google Drive แล้วโหลดจาก Drive
ครั้งแรกจะดาวน์โหลดไฟล์ไปเก็บที่ `MyDrive/CP_Project/data` ครั้งต่อไปจะโหลดจาก Drive เลย

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/CP_Project/data"
os.makedirs(DATA_DIR, exist_ok=True)
ZIP_PATH = f"{DATA_DIR}/KKU-EUOnlineRetail.csv.zip"
DATA_URL = "https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1/raw/main/dataset/KKU-EUOnlineRetail.csv.zip"

if not os.path.exists(ZIP_PATH):
    r = requests.get(DATA_URL, timeout=120)
    r.raise_for_status()
    open(ZIP_PATH, "wb").write(r.content)
    print("ดาวน์โหลดลง Drive แล้ว")
else:
    print("มีไฟล์ใน Drive แล้ว")

with zipfile.ZipFile(ZIP_PATH) as z:
    csv_name = [n for n in z.namelist() if n.endswith(".csv") and not n.startswith("__MACOSX")][0]
    df = pd.read_csv(z.open(csv_name), encoding="ISO-8859-1")   # ไฟล์มีเครื่องหมาย £ ต้องใช้ encoding นี้

print(df.shape)
df.head()

## 3. ดูข้อมูล (EDA)

In [ ]:
print(df.isna().sum())
print("ใบเสร็จยกเลิก (ขึ้นต้นด้วย C):", df["InvoiceNo"].astype(str).str.startswith("C").sum())
print("Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("UnitPrice <= 0:", (df["UnitPrice"] <= 0).sum())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["Country"].value_counts().head(10).plot.barh(ax=axes[0], title="Top 10 countries")
df["Description"].value_counts().head(10).plot.barh(ax=axes[1], title="Top 10 products")
plt.tight_layout(); plt.show()

## 4. ทำความสะอาดข้อมูล
1. ลบใบเสร็จยกเลิก (`InvoiceNo` ขึ้นต้นด้วย C)
2. ลบ `Quantity` และ `UnitPrice` ที่ ≤ 0
3. ลบแถวที่ไม่มีชื่อสินค้า และตัดช่องว่างหัวท้ายชื่อ
4. ลบ `POSTAGE` (ค่าส่ง ไม่ใช่สินค้า)

In [ ]:
clean = df.copy()
clean["InvoiceNo"] = clean["InvoiceNo"].astype(str)
clean = clean[~clean["InvoiceNo"].str.startswith("C")]
clean = clean[(clean["Quantity"] > 0) & (clean["UnitPrice"] > 0)]
clean = clean.dropna(subset=["Description"])
clean["Description"] = clean["Description"].str.strip()
clean = clean[~clean["Description"].isin(["POSTAGE", "DOTCOM POSTAGE", "Manual", "CARRIAGE"])]

print(f"ก่อน: {len(df):,} แถว → หลัง: {len(clean):,} แถว")

## 5. แปลงเป็นตะกร้าสินค้า (basket)
เลือกประเทศ `United Kingdom` เพราะข้อมูลเยอะที่สุด (เปลี่ยนเป็นประเทศอื่นได้)
แต่ละแถว = ใบเสร็จ 1 ใบ, แต่ละคอลัมน์ = สินค้า 1 ชิ้น, `True` = มีสินค้านี้ในใบเสร็จ

In [ ]:
COUNTRY = "United Kingdom"
transactions = clean[clean["Country"] == COUNTRY].groupby("InvoiceNo")["Description"].apply(lambda s: list(set(s)))

te = TransactionEncoder()
basket = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)
print(f"{COUNTRY}: {basket.shape[0]:,} ใบเสร็จ x {basket.shape[1]:,} สินค้า")
basket.iloc[:5, :6]

## 6. โมเดลที่ 1: Apriori
หาชุดสินค้าที่ถูกซื้อด้วยกันบ่อย (frequent itemsets)
`MIN_SUPPORT = 0.02` แปลว่าชุดสินค้านั้นต้องอยู่ในอย่างน้อย 2% ของใบเสร็จ

In [ ]:
MIN_SUPPORT = 0.02

start = time.time()
itemsets_apriori = apriori(basket, min_support=MIN_SUPPORT, use_colnames=True)
apriori_time = time.time() - start

print(f"Apriori: {len(itemsets_apriori)} itemsets ใน {apriori_time:.2f} วินาที")
itemsets_apriori.sort_values("support", ascending=False).head(10)

## 7. สร้างกฎ (Association Rules)
กฎหน้าตาแบบนี้: **ซื้อ A (antecedents) → มักซื้อ B (consequents)**

In [ ]:
rules = association_rules(itemsets_apriori, metric="lift", min_threshold=1)
rules = rules.sort_values("lift", ascending=False)
print("จำนวนกฎทั้งหมด:", len(rules))
rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

### 📖 อ่านค่ายังไง และควรดูค่าไหนเป็นหลัก

| ค่า | ความหมาย | ตัวอย่าง |
|---|---|---|
| **Support** | กฎนี้เกิดบ่อยแค่ไหน = สัดส่วนใบเสร็จที่มีทั้ง A และ B | 0.03 = 3% ของใบเสร็จมีทั้งคู่ |
| **Confidence** | ถ้าซื้อ A แล้ว มีโอกาสซื้อ B กี่ % | 0.70 = ลูกค้าที่ซื้อ A 70% ซื้อ B ด้วย |
| **Lift** | ซื้อ A แล้วทำให้โอกาสซื้อ B เพิ่มขึ้นกี่เท่าจากปกติ | 5 = เพิ่มขึ้น 5 เท่า |

**ดู Lift เป็นหลัก** แล้วค่อยดู Confidence กับ Support ประกอบ
1. **Lift > 1** แปลว่า A กับ B เกี่ยวกันจริง (ยิ่งสูงยิ่งเกี่ยวกันมาก), Lift = 1 คือบังเอิญ, Lift < 1 คือไม่ค่อยซื้อด้วยกัน
2. **Confidence** สูงพอ (เช่น ≥ 0.5) แปลว่ากฎนี้เกิดขึ้นบ่อยเมื่อลูกค้าซื้อ A
3. **Support** ไม่ต่ำเกินไป แปลว่ามีลูกค้าเยอะพอให้คุ้มทำโปร

⚠️ อย่าเรียงตาม Confidence อย่างเดียว เพราะถ้า B เป็นสินค้าขายดีอยู่แล้ว Confidence จะสูงเองแม้ A กับ B ไม่เกี่ยวกันเลย Lift แก้ปัญหานี้เพราะหารความนิยมของ B ออกไปแล้ว

In [ ]:
good_rules = rules[(rules["lift"] >= 3) & (rules["confidence"] >= 0.5) & (rules["support"] >= MIN_SUPPORT)]
print("กฎที่ดี (lift ≥ 3, confidence ≥ 0.5):", len(good_rules))
good_rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(15)

## 8. โมเดลที่ 2: FP-Growth
ได้ผลเหมือน Apriori ทุกตัว แต่เร็วกว่า เพราะบีบข้อมูลเป็นต้นไม้แล้วค้นหาจากต้นไม้ ไม่ต้องสแกนข้อมูลซ้ำหลายรอบ

In [ ]:
start = time.time()
itemsets_fp = fpgrowth(basket, min_support=MIN_SUPPORT, use_colnames=True)
fp_time = time.time() - start

print(f"FP-Growth: {len(itemsets_fp)} itemsets ใน {fp_time:.2f} วินาที")
print(f"Apriori  : {len(itemsets_apriori)} itemsets ใน {apriori_time:.2f} วินาที")
print("ได้ itemsets เหมือนกันไหม?", set(itemsets_fp["itemsets"]) == set(itemsets_apriori["itemsets"]))

pd.DataFrame({"Model": ["Apriori", "FP-Growth"], "Seconds": [apriori_time, fp_time]}) \
  .plot.bar(x="Model", y="Seconds", legend=False, title="Apriori vs FP-Growth: runtime", rot=0)
plt.show()

📖 **ควรดูอะไร:** จำนวน itemsets ต้องเท่ากัน (ยืนยันว่าถูก) แล้วดูเวลา ตัวไหนเร็วกว่าเลือกตัวนั้นใช้จริง

## 9. โมเดลที่ 3: Item2Vec
มองใบเสร็จเป็น "ประโยค" และสินค้าเป็น "คำ" แล้วฝึก Word2Vec ให้สินค้าที่มักอยู่ใบเดียวกันมี vector ใกล้กัน
ข้อดีคือแนะนำสินค้าที่ขายไม่บ่อยได้ด้วย (Apriori/FP-Growth ตัดสินค้าพวกนี้ทิ้งเพราะไม่ผ่าน min_support)

In [ ]:
sentences = [t for t in transactions if len(t) >= 2]
item2vec = Word2Vec(sentences, vector_size=50, window=10, min_count=5, sg=1, epochs=10, seed=42)
print("สินค้าที่มี vector:", len(item2vec.wv), "ตัว")

item = "WHITE HANGING HEART T-LIGHT HOLDER"
pd.DataFrame(item2vec.wv.most_similar(item, topn=5), columns=["สินค้าที่คล้ายกัน", "similarity"])

📖 **ควรดูอะไร:** `similarity` อยู่ระหว่าง 0–1 ยิ่งใกล้ 1 ยิ่งมักถูกซื้อในตะกร้าแบบเดียวกัน ใช้แนะนำสินค้าเพิ่มเมื่อหากฎไม่เจอ

## 10. กราฟกฎ

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(rules["support"], rules["confidence"], c=rules["lift"], cmap="viridis", alpha=0.7)
plt.colorbar(label="lift")
plt.xlabel("support"); plt.ylabel("confidence"); plt.title("Rules: support vs confidence (color = lift)")
plt.show()

📖 **ควรดูอะไร:** กฎที่ดีอยู่ **มุมขวาบนและสีเหลือง** (เกิดบ่อย, มั่นใจสูง, lift สูง)

## 11. นำไปใช้ #1: จัดโปรโมชันสินค้าคู่กัน
เลือกกฎ 1 ต่อ 1 ที่ดีที่สุด เพื่อทำโปร **"ซื้อ A ลด B"** หรือขายเป็นเซต

In [ ]:
pairs = good_rules[(good_rules["antecedents"].apply(len) == 1) & (good_rules["consequents"].apply(len) == 1)].copy()
pairs["ซื้อ A"] = pairs["antecedents"].apply(lambda x: list(x)[0])
pairs["แนะนำโปร B"] = pairs["consequents"].apply(lambda x: list(x)[0])
promotion = pairs.sort_values(["lift", "confidence"], ascending=False)[["ซื้อ A", "แนะนำโปร B", "support", "confidence", "lift"]]
promotion.head(10)

## 12. นำไปใช้ #2: จัดวางสินค้าในร้าน
วาดกราฟเครือข่าย: จุด = สินค้า, เส้น = มีกฎที่ดีเชื่อมกัน
สินค้าที่อยู่กลุ่มเดียวกันในกราฟ **ควรวางไว้ใกล้กัน** (ชั้นเดียวกัน หรือโซนเดียวกัน)

In [ ]:
G = nx.Graph()
for _, r in pairs.iterrows():
    G.add_edge(r["ซื้อ A"], r["แนะนำโปร B"], weight=r["lift"])

zones = list(nx.algorithms.community.greedy_modularity_communities(G))
for i, zone in enumerate(zones, 1):
    print(f"โซน {i}: {', '.join(list(zone)[:5])}")

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42, k=0.5)
colors = {item: i for i, zone in enumerate(zones) for item in zone}
nx.draw(G, pos, node_color=[colors[n] for n in G.nodes], cmap="tab10", node_size=300,
        with_labels=True, font_size=6, edge_color="lightgray")
plt.title("Product placement zones (same color = place together)")
plt.show()

## 13. ลองแนะนำสินค้า
ใส่สินค้าที่อยู่ในตะกร้า ระบบจะแนะนำจากกฎก่อน ถ้ากฎไม่เจอจะใช้ Item2Vec แทน

In [ ]:
def recommend(cart, k=5):
    cart = set(cart)
    found = rules[rules["antecedents"].apply(lambda a: a <= cart)]
    recs = []
    for c in found.sort_values("lift", ascending=False)["consequents"]:
        for item in c:
            if item not in cart and item not in recs:
                recs.append(item)
    if len(recs) < k:   # กฎไม่พอ ใช้ Item2Vec เติม
        known = [i for i in cart if i in item2vec.wv]
        if known:
            recs += [i for i, _ in item2vec.wv.most_similar(known, topn=k * 2) if i not in cart and i not in recs]
    return recs[:k]

recommend(["JUMBO BAG RED RETROSPOT", "LUNCH BAG RED RETROSPOT"])

## 14. สรุปค่าที่ต้องดู
| ส่วน | ดูค่าอะไร | เกณฑ์ |
|---|---|---|
| Apriori / FP-Growth | จำนวน itemsets เท่ากัน + เวลา | เลือกตัวที่เร็วกว่า |
| กฎ (Rules) | **Lift เป็นหลัก** → Confidence → Support | Lift > 1 (ดีถ้า ≥ 3), Confidence ≥ 0.5 |
| Item2Vec | similarity | ใกล้ 1 = มักซื้อด้วยกัน |
| โปรโมชัน | กฎ 1 ต่อ 1 ที่ Lift และ Confidence สูง | ทำโปรซื้อ A ลด B |
| จัดวางสินค้า | โซนจากกราฟเครือข่าย | สินค้าโซนเดียวกันวางใกล้กัน |